## Modified segmentation mask
- Used for pix2pixRAD and custom diffusion training
- segmentation masks originally contain values [0,3]
- healthy tissue added as value 4
- subsequent masks scaled to [0,255]

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
from skimage import morphology


HEALTHY_LABEL = 4
MAX_VALUE_OF_CLASS = 4      # new max of seg after healthy tissue
T1C_THRESHOLD = 1           # represents brain tissue in T1c scans

split_root = Path('path/to/your/split/data/root')

# -----------------------------
# Process each split
# -----------------------------
for split_name in ['train', 'test', 'validation']:
    split_dir = split_root / split_name
    patient_folders = sorted([p for p in split_dir.iterdir() if p.is_dir()])

    for patient in patient_folders:
        seg_dir = patient / 'seg'
        seg_mod_dir = patient / 'seg_mod'
        t1c_dir = patient / 't1c'  

        if not seg_dir.is_dir():
            print(f' {patient.name} — missing seg folder.')
            continue

        seg_mod_dir.mkdir(parents=True, exist_ok=True)

        t1c_files = sorted(t1c_dir.glob('*.png'))
        seg_files = sorted(seg_dir.glob('*.png'))

        for t1c_file, seg_file in zip(t1c_files, seg_files):
            
            t1c_img = np.array(Image.open(t1c_file)).astype('float32')
            seg_img = np.array(Image.open(seg_file)).astype('float32')

            # make sure to normalize T1c (to ensure healthy tissue is correctly derived)
            if t1c_img.max() > t1c_img.min():
                t1c_norm = (t1c_img - t1c_img.min()) / (t1c_img.max() - t1c_img.min())
            else:
                t1c_norm = np.zeros_like(t1c_img)
            t1c_norm = (t1c_norm * 255).astype('uint8')

            # create brain mask from norm T1c
            thresh = (t1c_norm > T1C_THRESHOLD).astype(np.uint8)
            thresh = morphology.binary_erosion(thresh, morphology.square(3))
            thresh = morphology.binary_closing(thresh, morphology.square(6))

            # fill relevant 0s in seg with healthy label of 4, derived from T1c
            seg_mod = np.where(seg_img == 0, thresh * HEALTHY_LABEL, seg_img)

            # scale to [0, 255]
            seg_mod = (seg_mod / MAX_VALUE_OF_CLASS * 255).astype('uint8')

            # save new seg_mod images
            Image.fromarray(seg_mod).save(seg_mod_dir / seg_file.name)

        print(f'  [{split_name}] Processed {patient.name} — {len(t1c_files)} slices.')

print('Done')

## Scaled segmentation mask
- used for SynDiff training
- identical scaling to above, except no healthy tissue added

In [ ]:

MAX_VALUE_OF_CLASS = 3      # max of seg 

split_root = Path('path/to/your/split/root')

# -----------------------------
# Process each split
# -----------------------------
for split_name in ['train', 'test', 'validation']:
    split_dir = split_root / split_name
    patient_folders = sorted([p for p in split_dir.iterdir() if p.is_dir()])

    for patient in patient_folders:
        seg_dir = patient / 'seg'
        seg_scaled_dir = patient / 'seg_scaled'

        if not seg_dir.is_dir():
            print(f' {patient.name} — missing seg folder.')
            continue

        seg_scaled_dir.mkdir(parents=True, exist_ok=True)

        seg_files = sorted(seg_dir.glob('*.png'))

        for seg_file in seg_files:
            
            seg_img = np.array(Image.open(seg_file)).astype('float32')

            seg_scaled = (seg_img / MAX_VALUE_OF_CLASS * 255).astype('uint8')

            # save new seg_mod images
            Image.fromarray(seg_scaled).save(seg_scaled_dir / seg_file.name)

        print(f'  [{split_name}] Processed {patient.name} .')

print('Done')